# Register and serve an OSS embedding model (EmbeddingGemma 2)

This notebook sets up the `google/embeddinggemma-2` embedding model in a Model Serving endpoint usable for Vector Search.
* (Optional) Log in to Hugging Face Hub.
* Download the model from the Hugging Face Hub.
* Register it to the MLflow Model Registry (Unity Catalog).
* Start a GPU Model Serving endpoint to serve the model.

The model is available at https://huggingface.co/google/embeddinggemma-2.
* Text-only configuration: 270M parameters (vision and audio encoders disabled), 768-dimensional embeddings (same as embeddinggemma-300m)
* Based on the Gemma architecture, 8,192-token context
* Apache 2.0 license
* Loaded in float32: float16 produces NaNs/degraded embeddings, and the T4 GPU (GPU_SMALL) has no native bfloat16 support

**Vector Search compatibility**: The model is logged with the `mlflow.sentence_transformers` flavor, producing embeddings in the standard format expected by Databricks Vector Search managed embedding indexes.

## Install Databricks Python SDK

This notebook uses its Python client to work with serving endpoints.

In [0]:
%pip install -U databricks-sdk python-snappy "protobuf>=5.29.5,<6"
%pip install -U sentence-transformers[image] transformers>=5.11.0 torch accelerate
dbutils.library.restartPython()

## Download model

In [0]:
# # (Optional) Log in to Hugging Face Hub. embeddinggemma-2 is Apache 2.0; login avoids anonymous rate limits.
# from huggingface_hub import login

# # HF token is stored in Databricks secrets (scope: justinm-ai-secrets, key: hf_pat)
# hf_token = dbutils.secrets.get(scope="justinm-ai-secrets", key="hf_pat")
# # hf_token = "your_hf_token"  # Replace with your HF token
# login(token=hf_token)

# # After running this cell, authenticated downloads are used in subsequent code cells.

In [0]:
# Download model using the sentence_transformers library.
import torch
from sentence_transformers import SentenceTransformer

source_model_name = 'google/embeddinggemma-2'  # model name on Hugging Face Hub

# Text-only: disable vision and audio encoders (270M params vs 740M full multimodal).
# config_kwargs are passed to AutoConfig.from_pretrained(), overriding loaded config attributes.
# Force float32: float16 yields NaNs for EmbeddingGemma 2, and T4 GPUs lack native bfloat16.
model = SentenceTransformer(
    source_model_name,
    model_kwargs={"dtype": torch.float32},
    config_kwargs={"vision_config": None, "audio_config": None},
)

In [0]:
# Test the model, just to show it works.
sentences = ["This is an example sentence", "Each sentence is converted"]
embeddings = model.encode(sentences)
print(f"Embedding shape: {embeddings.shape}")  # (num_sentences, embedding_dim)
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"First embedding (truncated): {embeddings[0][:10]}...")

## Register model to MLflow

In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

# Set the region if this is on AWS GovCloud
import os
os.environ["AWS_DEFAULT_REGION"] = "us-gov-west-1"

In [0]:

# Specify the catalog and schema to use. You must have USE_CATALOG privilege on the catalog and USE_SCHEMA and CREATE_TABLE privileges on the schema.
# Change the catalog and schema here if necessary.
catalog = "justinm"
schema = "embeddings"
model_name = "embeddinggemma-2"

In [0]:
# MLflow model name. The Model Registry uses this name for the model.
registered_model_name = f"{catalog}.{schema}.{model_name}"

In [0]:
# Compute input and output schema.
signature = mlflow.models.signature.infer_signature(sentences, embeddings)
print(signature)

In [0]:
import tempfile, os
import pandas as pd
import numpy as np

# Custom pyfunc wrapper to properly handle DataFrame input from Model Serving.
# The sentence_transformers flavor's built-in pyfunc passes a pandas Series to
# model.encode(), which expects a list of strings.
# This wrapper extracts strings from the DataFrame before calling encode().
class EmbeddingModelWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import torch
        from sentence_transformers import SentenceTransformer
        # float32 on T4 (GPU_SMALL): no native bf16, and fp16 produces NaNs for this model.
        device = "cuda" if torch.cuda.is_available() else "cpu"
        # Text-only: disable vision/audio encoders.
        self.model = SentenceTransformer(
            context.artifacts["model_dir"],
            device=device,
            model_kwargs={"dtype": torch.float32},
            config_kwargs={"vision_config": None, "audio_config": None},
        )

    def predict(self, context, model_input, params=None):
        if isinstance(model_input, pd.DataFrame):
            sentences = model_input.iloc[:, 0].tolist()
        elif isinstance(model_input, np.ndarray):
            sentences = model_input.tolist()
        elif isinstance(model_input, list):
            sentences = model_input
        else:
            sentences = list(model_input)
        embeddings = self.model.encode(sentences)
        return embeddings

# Save the downloaded model to a temp directory for artifact logging
model_dir = os.path.join(tempfile.mkdtemp(), "embeddinggemma-2")
model.save(model_dir)

# Log with custom pyfunc wrapper
model_info = mlflow.pyfunc.log_model(
    artifact_path="model",
    python_model=EmbeddingModelWrapper(),
    artifacts={"model_dir": model_dir},
    signature=signature,
    input_example=sentences,
    registered_model_name=registered_model_name,
    pip_requirements=[
        "sentence-transformers[image]",
        "transformers>=5.11.0",
        "torch",
        "accelerate",
    ],
)

In [0]:
inference_test = ["I enjoy pies of both apple and cherry.", "I prefer cookies."]

# Load as pyfunc (same path Model Serving uses) to validate the custom wrapper.
loaded_model_pyfunc = mlflow.pyfunc.load_model(model_info.model_uri)

# Test with a DataFrame input — this is exactly what Model Serving passes to predict().
import pandas as pd
test_df = pd.DataFrame(inference_test)
embeddings_test = loaded_model_pyfunc.predict(test_df)
print(f"Output shape: {embeddings_test.shape}")
print(f"Embedding dim: {embeddings_test.shape[1]}")
embeddings_test

In [0]:
# Extract the version of the model you just registered.
mlflow_client = mlflow.MlflowClient()

def get_latest_model_version(model_name):
  client = mlflow_client
  model_version_infos = client.search_model_versions("name = '%s'" % model_name)
  return max([int(model_version_info.version) for model_version_info in model_version_infos])

model_version = get_latest_model_version(registered_model_name)
model_version

### Create model serving endpoint

For more details, see "Create foundation model serving endpoints" ([AWS](https://docs.databricks.com/en/machine-learning/model-serving/create-foundation-model-endpoints) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/create-foundation-model-endpoints/) | [GCP](https://docs.databricks.com/gcp/en/machine-learning/model-serving/create-foundation-model-endpoints)).

**Note**: This example creates a GPU_SMALL endpoint (1x NVIDIA T4, 16 GB on AWS). The text-only EmbeddingGemma 2 model is 270M parameters (~1 GB in float32), fitting easily on a T4. Cold start may take 10-15 minutes.

In [0]:
endpoint_name = "embeddinggemma-2"  # Name of endpoint to create

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput

w = WorkspaceClient()

In [0]:
endpoint_config_dict = {
    "served_entities": [
        {
            "name": endpoint_name,
            "entity_name": registered_model_name,
            "entity_version": model_version,
            "workload_type": "GPU_SMALL",
            "workload_size": "Small",
            "scale_to_zero_enabled": False,
        }
    ]
}

from databricks.sdk.errors import ResourceDoesNotExist
from databricks.sdk.service.serving import ServedEntityInput, ServingModelWorkloadType

served_entities = [
    ServedEntityInput(
        name=endpoint_name,
        entity_name=registered_model_name,
        entity_version=str(model_version),
        workload_type=ServingModelWorkloadType.GPU_SMALL,
        workload_size="Small",
        scale_to_zero_enabled=False,
    )
]

from datetime import timedelta

# Create endpoint if it doesn't exist, otherwise update it with new model version.
# GPU_SMALL endpoints can take 20-30 min for cold start; set timeout accordingly.
try:
    w.serving_endpoints.get(name=endpoint_name)
    print(f"Endpoint '{endpoint_name}' exists. Updating to model version {model_version}...")
    w.serving_endpoints.update_config_and_wait(
        name=endpoint_name,
        served_entities=served_entities,
        timeout=timedelta(minutes=40),
    )
except ResourceDoesNotExist:
    print(f"Creating endpoint '{endpoint_name}'...")
    endpoint_config = EndpointCoreConfigInput.from_dict(endpoint_config_dict)
    w.serving_endpoints.create_and_wait(name=endpoint_name, config=endpoint_config, timeout=timedelta(minutes=40))

print("Endpoint ready.")

### Query endpoint

The above `create_and_wait` command waits until the endpoint is ready. You can also check the status of the serving endpoint in the Databricks UI.

For more information, see "Query foundation models" ([AWS](https://docs.databricks.com/aws/en/machine-learning/model-serving/score-foundation-models) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/score-foundation-models/) | [GCP](https://docs.databricks.com/gcp/en/machine-learning/model-serving/score-foundation-models)).

In [0]:
# Only run this command after the Model Serving endpoint is in the Ready state.
import time

start = time.time()

# If the endpoint is not yet ready, you might get a timeout error. If so, wait and then rerun the command.
endpoint_response = w.serving_endpoints.query(name=endpoint_name, dataframe_records=['Hello world', 'Good morning'])

end = time.time()

print(endpoint_response)
print(f'Time taken for querying endpoint in seconds: {end-start}')